In [1]:
%matplotlib widget

from blackjack_py import ProbabilisticRankShoe


from blackjack.blackjack_round import BJRound, BJStage, BJRules
from blackjack.actions import PlayerAction, DealerAction
from blackjack.cards import Card, Rank
import numpy as np
import time
from datetime import datetime
import os
import tqdm
from collections import deque
# from blackjack.shoe import ProbabilisticRankShoe
import time
from blackjack.floor_ceil_node import FloorCeilNode, SplitNode, DecisionNode, DealerCheckBJNode
from blackjack.dealer_sim import run_dealer_cards_simulation
import datetime
import matplotlib.pyplot as plt
from blackjack.tree_utils import iterate_nodes_by_levels
import os
from blackjack import dealer_sim


In [2]:
rules = BJRules(
    dealer_checks_blackjack=True,
    dealer_hits_soft_17=False,
    allow_late_surrender=False,
    allow_early_surrender_on_ten=False,
    allow_early_surrender_on_ace=False,
    allow_early_surrender_on_all=False,
    dealer_shows_card_on_surrender=False,
    allow_insurance_vs_ace=True,
    natural_blackjack_payout=3/2,
    surrender_payout=1/2,
    insurance_payout=2/1,
    max_splits_allowed=1,
    allow_action_on_split_aces=True,
    allow_double_after_split=True,
    allow_double_on_soft=True,
    allow_split_different_tens=True
)

In [3]:
def generate_initial_rounds():
    for player_card_0 in range(2, 12):
        for player_card_1 in range(player_card_0, 12):
            for dealer_upcard in range(2, 12):
                n_count = 1 if player_card_0 == player_card_1 else 2
                yield player_card_0, player_card_1, dealer_upcard, n_count

In [4]:
def build_tree(root_node: FloorCeilNode, gap_target=0.1):
    gap_target_unit = gap_target * root_node.bj_round.bet_unit 
    root_node.build_tree()
    for i in range(100):
        t0 = time.time()
        root_node.convert_to_full_up_to_depth(depth=i)
        t1 = time.time()
        if root_node.get_ceil_value() - root_node.get_floor_value() < gap_target_unit:
            return
    raise RuntimeError("Failed to converge")

In [5]:
nodes = []
probs = []
for p0, p1, d, n in generate_initial_rounds():
    shoe = ProbabilisticRankShoe.seeded(8, 42)
    bj_round = BJRound(rules)
    bj_round.start_round(100)

    prob = 1
    for c in [p0, p1, d]:
        prob_c_dict = shoe.get_rank_value_probabilities()
        prob *= prob_c_dict[c]
        bj_round.take_card(c)
        shoe.burn_rank_value(d)

    root_node = DecisionNode(
        bj_round,
        shoe,
        max_hand_size_full_enum=1,
        dealer_sim_depth=4,
        sim_algo="combo"
    )
    nodes.append(root_node)
    probs.append(n * prob)

In [ ]:
bj_round = BJRound(rules)
shoe = ProbabilisticRankShoe.seeded(8, 42)
bj_round.start_round(10)

cards = [
    Card(Rank.ACE),
    Card(Rank.ACE),
    Card(Rank.ACE) 
]

# cards = [
#     Card(Rank.TEN),
#     Card(Rank.SIX),
#     Card(Rank.ACE)  
# ]

cards = [c.rank_value() for c in cards]

bj_round.take_card(cards[0])
bj_round.take_card(cards[1])
bj_round.take_card(cards[2])

shoe.burn_rank_value(cards[0])
shoe.burn_rank_value(cards[1])
shoe.burn_rank_value(cards[2])

root_node_fe1 = DecisionNode(
    bj_round,
    shoe,
    max_hand_size_full_enum=1
)


root_node_fe3 = DecisionNode(
    bj_round,
    shoe,
    max_hand_size_full_enum=3
)

print(str(bj_round))

Last card: 7
Dealer 7(7)
Player A,A(2/12)[$10]


In [ ]:
runtime_data = {}

for depth in tqdm.tqdm([2, 3, 4, 5, 6, 7]):
    for algo in ["combo", "recursive"]:
        root_node_timing = DecisionNode(
            bj_round,
            shoe,
            max_hand_size_full_enum=1,
            dealer_sim_depth=depth,
            sim_algo=algo
        )
        t0 = time.time()
        
        root_node_timing.build_tree()
        for i in range(100):
            t0 = time.time()
            root_node_timing.convert_to_full_up_to_depth(depth=i)
            t1 = time.time()
            if root_node_timing.get_ceil_value() - root_node_timing.get_floor_value() < 0.01:
                break

        t1 = time.time()
        gap = (root_node_timing.get_floor_value(), root_node_timing.get_ceil_value())
        runtime = t1 - t0

        runtime_data[(depth, algo)] = (runtime, gap)

100%|██████████| 6/6 [04:00<00:00, 40.08s/it]


In [ ]:
target_gap = runtime_data[(7, "recursive")][1]
target_rec = (target_gap[0] + target_gap[1]) / 2

target_gap = runtime_data[(7, "combo")][1]
target_comb = (target_gap[0] + target_gap[1]) / 2

target = (target_rec + target_comb) / 2

In [ ]:
# Print results as a formatted table
print("=" * 70)
print(f"{'Depth':<8} {'Algorithm':<12} {'Runtime (s)':<14} {'Floor':<10} {'Ceil':<10} {'Err':<10}")
print("=" * 70)

for depth in [2, 3, 4, 5, 6, 7]:
    for algo in ["combo", "recursive"]:
        runtime, (floor_val, ceil_val) = runtime_data[(depth, algo)]
        mid = (ceil_val + floor_val) / 2
        err = mid - target
        print(f"{depth:<8} {algo:<12} {runtime:<14.3f} {floor_val:<10.4f} {ceil_val:<10.4f} {err:<10.4f}")
    print("-" * 70)

print("=" * 70)

Depth    Algorithm    Runtime (s)    Floor      Ceil       Err       
2        combo        2.006          7.4461     7.4531     -1.7217   
2        recursive    2.028          9.1757     9.1844     0.0087    
----------------------------------------------------------------------
3        combo        4.119          8.9312     8.9392     -0.2361   
3        recursive    5.109          9.2013     9.2094     0.0341    
----------------------------------------------------------------------
4        combo        6.517          9.1498     9.1579     -0.0175   
4        recursive    9.509          9.1685     9.1766     0.0013    
----------------------------------------------------------------------
5        combo        8.478          9.1666     9.1747     -0.0007   
5        recursive    13.088         9.1672     9.1754     -0.0000   
----------------------------------------------------------------------
6        combo        9.791          9.1672     9.1753     -0.0000   
6        recursi

In [ ]:
Last card: 4
Dealer 4(4)
Player A,A(2/12)[$10]
100%|██████████| 6/6 [13:32<00:00, 135.37s/it]
======================================================================
Depth    Algorithm    Runtime (s)    Floor      Ceil       Err       
======================================================================
2        combo        2.682          4.3071     4.3081     3.8813    
2        recursive    3.827          8.2277     8.2302     0.0401    
----------------------------------------------------------------------
3        combo        7.668          7.4545     7.4568     0.7332    
3        recursive    15.199         8.2031     8.2057     0.0156    
----------------------------------------------------------------------
4        combo        14.193         8.1085     8.1111     0.0790    
4        recursive    34.571         8.1886     8.1913     0.0011    
----------------------------------------------------------------------
5        combo        21.596         8.1823     8.1849     0.0053    
5        recursive    61.566         8.1874     8.1900     0.0002    
----------------------------------------------------------------------
6        combo        28.216         8.1874     8.1900     0.0002    
6        recursive    89.164         8.1876     8.1902     0.0000    
----------------------------------------------------------------------
7        combo        35.301         8.1876     8.1902     0.0000    
7        recursive    97.196         8.1876     8.1902     0.0000    
----------------------------------------------------------------------
======================================================================

SyntaxError: invalid character '█' (U+2588) (3363792463.py, line 4)

In [ ]:
Seed 42
Last card: 7
Dealer 7(7)
Player 10,2(12)[$10]
  0%|          | 0/6 [00:00<?, ?it/s]
100%|██████████| 6/6 [00:04<00:00,  1.48it/s]
======================================================================
Depth    Algorithm    Runtime (s)    Floor      Ceil       Err       
======================================================================
2        combo        0.072          -2.5300    -2.5269    -0.4022   
2        recursive    0.063          -2.0176    -2.0138    0.1105    
----------------------------------------------------------------------
3        combo        0.167          -2.1867    -2.1831    -0.0587   
3        recursive    0.106          -2.1191    -2.1154    0.0089    
----------------------------------------------------------------------
4        combo        0.133          -2.1325    -2.1288    -0.0045   
4        recursive    0.195          -2.1274    -2.1237    0.0007    
----------------------------------------------------------------------
5        combo        0.171          -2.1282    -2.1245    -0.0002   
5        recursive    0.247          -2.1280    -2.1243    0.0000    
----------------------------------------------------------------------
6        combo        0.190          -2.1280    -2.1243    -0.0000   
6        recursive    0.293          -2.1280    -2.1243    -0.0000   
----------------------------------------------------------------------
7        combo        0.206          -2.1280    -2.1243    0.0000    
7        recursive    0.303          -2.1280    -2.1243    0.0000    
----------------------------------------------------------------------
======================================================================

In [ ]:
root_node_timing.decision_choice

In [ ]:
import pandas as pd

# Convert to DataFrame for nice display
rows = []
for (depth, algo), (runtime, (floor_val, ceil_val)) in runtime_data.items():
    rows.append({
        'Depth': depth,
        'Algorithm': algo,
        'Runtime (s)': f"{runtime:.3f}",
        'Floor': f"{floor_val:.4f}",
        'Ceil': f"{ceil_val:.4f}",
        'Gap': f"{ceil_val - floor_val:.4f}"
    })

df = pd.DataFrame(rows)
df_pivot = df.pivot(index='Depth', columns='Algorithm', values=['Runtime (s)', 'Gap'])
display(df_pivot)

In [ ]:
def build_tree(root_node):
    i = 0
    while True:
        t0 = time.time()
        root_node.build_tree_layer(depth=i)
        t1 = time.time()
        seconds = np.round(t1 - t0)
        dt_whole = datetime.timedelta(seconds=seconds)
        print(f"Depth {i} built in {dt_whole} ({seconds} s)")
        if root_node.tree_completed():
            print("Tree completed")
            break
        i += 1

In [ ]:
dealer_sim.total_sim_time = 0
t0 = time.time()
build_tree(root_node_fe1)
t1 = time.time()
print(f"Simulation time {dealer_sim.total_sim_time} seconds ")
print(f"Time taken to build tree for root_node_fe3: {t1 - t0} seconds")
print(f"{dealer_sim.total_sim_time / (t1 - t0) * 100:.02f}%")

In [ ]:
for i in range(100):
    dealer_sim.total_sim_time = 0
    t0 = time.time()
    root_node_fe1.convert_to_full_up_to_depth(depth=i)
    t1 = time.time()
    print(f"Depth {i} Simulation time {dealer_sim.total_sim_time} seconds ")
    print(f"Gap {root_node_fe1.get_ceil_value()}, {root_node_fe1.get_floor_value()}")
    print(f"Simulation time {dealer_sim.total_sim_time} seconds ")
    print(f"Time taken to build tree for root_node_fe1: {t1 - t0} seconds")
    print(f"{dealer_sim.total_sim_time / (t1 - t0) * 100:.02f}%")

    if root_node_fe1.get_ceil_value() - root_node_fe1.get_floor_value() < 0.01:
        break

In [ ]:
dealer_sim.total_sim_time = 0
t0 = time.time()
build_tree(root_node_fe3)
t1 = time.time()
print(f"Simulation time {dealer_sim.total_sim_time} seconds ")
print(f"Time taken to build tree for root_node_fe3: {t1 - t0} seconds")
print(f"{dealer_sim.total_sim_time / (t1 - t0) * 100:.02f}%")

In [ ]:
for i in range(100):
    dealer_sim.total_sim_time = 0
    t0 = time.time()
    root_node_fe3.convert_to_full_up_to_depth(depth=i)
    t1 = time.time()
    print(f"Depth {i} Simulation time {dealer_sim.total_sim_time} seconds ")
    print(f"Gap {root_node_fe3.get_ceil_value()}, {root_node_fe3.get_floor_value()}")
    print(f"Simulation time {dealer_sim.total_sim_time} seconds ")
    print(f"Time taken to build tree for root_node_fe3: {t1 - t0} seconds")
    print(f"{dealer_sim.total_sim_time / (t1 - t0) * 100:.02f}%")

    if root_node_fe3.get_ceil_value() - root_node_fe3.get_floor_value() < 0.01:
        break

In [ ]:
print(root_node_fe1.get_floor_value(), root_node_fe1.get_value(), root_node_fe1.get_ceil_value())
print(root_node_fe3.get_floor_value(), root_node_fe3.get_value(), root_node_fe3.get_ceil_value())


In [ ]:
# 7.19-7.20 - depth 2
# 7.206-7.203 - depth 3
# 9.136-9.1374 depth 4 - 18s for comb algo - 33s for recursive algo (9.165-9.167)
# 9.1636 - 9.1650 depth 5 - 28s for comb algo - 53s for recursive algo
# 9.165 - 9.166 depth 6

In [ ]:
abstract_node.total_sim_time

In [ ]:
log_tree_structure(root_node_fe1)
inspect(root_node_fe1)

In [ ]:
log_tree_structure(root_node_fe3)

inspect(root_node_fe3)

In [ ]:
print(root_node_fe1.get_floor_value(), root_node_fe1.get_value(), root_node_fe1.get_ceil_value())

In [ ]:
inspect(root_node_fe1)

In [ ]:
inspect(root_node_fe3)

In [ ]:
# lets compare split->card 8 node

In [ ]:
split_8_node_fe1 = root_node_fe1.children[1].children[1].children[3].children[6]
split_8_node_fe3 = root_node_fe3.children[1].children[1].children[3].children[6]

In [ ]:
log_tree_structure(split_8_node_fe1)
log_tree_structure(split_8_node_fe3)

In [ ]:
inspect(split_8_node_fe1)

In [ ]:
inspect(split_8_node_fe3)

In [ ]:
round = split_8_node_fe3.bj_round.copy()
round.take_action(PlayerAction.STAND)

results_100 = []
results_1000 = []
results_100_2 = []
results_100_3 = []
results_10_3 = []


n_runs = 3600

t0 = time.time()
for i in range(n_runs // 2):
    result_10_3 = run_dealer_cards_simulation_alt(
        round, split_8_node_fe3.shoe, n_dealer_sim_runs=10, n_full_sample=3
    )
    results_10_3.append(result_10_3)

t1 = time.time()
for i in range(n_runs // 2):
    result_100_3 = run_dealer_cards_simulation_alt(
        round, split_8_node_fe3.shoe, n_dealer_sim_runs=10, n_full_sample=3
    )
    results_100_3.append(result_100_3)

t2 = time.time()
for i in range(n_runs // 6):
    result_100_2 = run_dealer_cards_simulation_alt(
        round, split_8_node_fe3.shoe, n_dealer_sim_runs=100, n_full_sample=2
    )
    results_100_2.append(result_100_2)

t3 = time.time()
for i in range(n_runs // 12):
    result_1000 = run_dealer_cards_simulation_alt(
        round, split_8_node_fe3.shoe, n_dealer_sim_runs=1000, n_full_sample=1
    )
    results_1000.append(result_1000)

t4 = time.time()
for i in range(n_runs):
    result_100 = run_dealer_cards_simulation_alt(
        round, split_8_node_fe3.shoe, n_dealer_sim_runs=100
    )
    results_100.append(result_100)
t5 = time.time()

print(f"10 runs 3 levels time: {t1 - t0} s")
print(f"100 runs 3 levels time: {t2 - t1} s")
print(f"100 runs 2 levels time: {t3 - t2} s")
print(f"1000 runs time: {t4 - t3} s")
print(f"100 runs time: {t5 - t4} s")


In [ ]:
f, ax = plt.subplots(1,1)
# ax.hist(results_100, label="sim 100", alpha=0.5, density=True)
# ax.hist(results_1000, label="sim 1000",  alpha=0.5, density=True)
ax.hist(results_100_2, label="sim 100 2 lvl",  alpha=0.5, density=True)
ax.hist(results_100_3, label="sim 100 3 lvl",  alpha=0.5, density=True)
ax.hist(results_10_3, label="sim 10 3 lvl",  alpha=0.5, density=True)
ax.legend()

In [ ]:
[ch.get_value() for ch in root_node.children]

In [ ]:
root_node.children_events

In [ ]:
log_tree_structure(root_node)

In [ ]:
from blackjack.floor_ceil_node import DealerCheckBJNode

ceil = []
ev = []
floor = []
runtime = []

t_start = time.time()

root_node = root_node_fe1

for i in range(100):
    t0 = time.time()
    children_changed = root_node.convert_to_full_up_to_depth(depth=i)
    t1 = time.time()
    seconds = np.round(t1 - t0)

    floor_val = root_node.get_floor_value()
    val = root_node.get_value()
    ceil_val = root_node.get_ceil_value()

    floor.append(floor_val)
    ev.append(val)
    ceil.append(ceil_val)
    runtime.append(t1 - t_start)

    dt_whole = datetime.timedelta(seconds=seconds)
    print(f"Depth {i} converted to full enumeration in {dt_whole} ({seconds} s)")
    print(f"Value gap: [{floor_val:.02f} {val:.02f} {ceil_val:.02f}]")

    if (ceil_val - floor_val) < 0.001:
        break
    # log_tree_structure(root_node)

In [ ]:
f, ax = plt.subplots(figsize=(10, 6))
ax.plot(runtime, floor, label="Floor Value", linestyle='--', color="grey" )
ax.plot(runtime, ev, label="Expected Value", color="red")
ax.plot(runtime, ceil, label="Ceil Value", linestyle='--', color='grey')

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.legend()

In [ ]:
# starting from initial limit of 1 to depth 5 is a reasonable approach

In [ ]:
refuse_insurance_node = root_node.children[1]
print(refuse_insurance_node.children_prob)

In [ ]:
print([ch.get_floor_value() for ch in refuse_insurance_node.children])
print([ch.get_ceil_value() for ch in refuse_insurance_node.children])

In [ ]:
refuse_insurance_node.floor_value, refuse_insurance_node.ceil_value

In [ ]:
refuse_insurance_node.recompute_tree_value()

In [ ]:
refuse_insurance_node.floor_value, refuse_insurance_node.ceil_value

In [ ]:
lvl25 = None
for lvl, node in iterate_nodes_by_levels(root_node):
    if lvl == 24 and isinstance(node, DecisionNode):
        lvl25 = node
        break

In [ ]:
print(str(lvl25.bj_round))

In [ ]:
# with open("logs/game_tree.json", "w") as f:
#     game_tree_to_json(f, root_node)

In [ ]:
print(root_node.get_value())
print(root_node.children_prob)
print(root_node.children_events)
print([f"{ch.get_value():.2f}" for ch in root_node.children])

In [ ]:
node = root_node.children[-1].children[7]
print("decision", node.decision_choice)
print(str(node.bj_round))
print("value = ", node.get_value())
print(node.children_prob)
print(node.children_events)

node.decision_action = None
for ch in node.children:
    ch.decision_action = None
    
print([f"{ch.get_value():.2f}" for ch in node.children])
print([f"{ch.get_ceil_value():.2f}" for ch in node.children])
print([f"{ch.get_floor_value():.2f}" for ch in node.children])

In [ ]:
node.update_decision()

In [ ]:
node.children[1].get_value()

In [ ]:
print(node.children[1].cards_sampled) 
print(node.children[1].cards_not_sampled)
print(node.children[1].cards_21)
print(node.children[1].cards_bust)

In [ ]:
print(set(node.children[1].children_prob))

In [ ]:
node_child = node.children[1]

print(node_child.children_prob)
print(node_child.children_events)
print([f"{ch.get_value():.2f}" for ch in node_child.children])


In [ ]:
np.array(node_child.children_prob).dot(np.array([ch.get_value() for ch in node_child.children]))

In [ ]:
for ch, ch_event in zip(node.children, node.children_events):
    print(ch_event)
    log_tree_structure(ch)

In [ ]:
node.rebuild_children()
node.build_tree()
node.convert_to_full_up_to_depth(np.inf)

In [ ]:
print(node.children_events)
print([f"{ch.get_value():.2f}" for ch in node.children])

for ch, ch_event in zip(node.children, node.children_events):
    print(ch_event)
    log_tree_structure(ch)

In [ ]:
root_node.convert_to_full_up_to_depth(depth=np.inf)

In [ ]:
values = []

values.append(root_node.get_value())

for i in tqdm.tqdm(range(1000)):
    root_node.resample_player_cards()
    values.append(root_node.get_value())
mean_value = np.mean(values)


In [ ]:
f, ax = plt.subplots()
ax.scatter(range(len(values)), values)
ax.hlines(mean_value, xmin=0, xmax=len(values), color="red", label=f"mean={mean_value:.3f}")
ax.legend()

In [ ]:
np.min(values), np.max(values)

In [ ]:
for lvl in range(25):
    level_nodes = get_nodes_on_the_level(root_node, lvl)
    print(f"Level {lvl} has {len(level_nodes)} nodes")
    finished = False
    for n in level_nodes:
        if n.bj_round.get_stage() == BJStage.ROUND_OVER:
            continue
        elif isinstance(n, MonteCarloNode):
            print(f"MonteCarloNode found on level {lvl}")
            finished = True
            break
        else:
            break
    
    if finished:
        break

In [ ]:
from collections import Counter
count = Counter()

player_card_nodes = []
player_action_nodes = []
for node in level_nodes:
    stage = node.bj_round.get_stage()
    count[stage] += 1
    if stage == BJStage.PLAYER_CARD:
        player_card_nodes.append(node)
    if stage == BJStage.PLAYER_ACTION:
        player_action_nodes.append(node)



In [ ]:
count

In [ ]:
for i, n in enumerate(player_action_nodes):
    print(f"idx = {i}")
    print(f"Value = {n.get_value()}")
    print(str(n.bj_round))
    print()

In [ ]:
this_node = player_card_nodes[-1].parent
this_node.build_tree()

print(f"this_node Value = {this_node.get_value()}")
print(str(this_node.bj_round))
first_value = this_node.get_value()

In [ ]:
for i in range(100):
    changed = this_node.resample_player_cards()
    if this_node.get_value() == first_value:
        continue
    print(changed, this_node.get_value())
    print("-" * 32)

    # for i in range(len(this_node.children)):
    #     p = this_node.children_prob[i]
    #     ch = this_node.children[i]
    #     print(f"Value = {ch.get_value()}")
    #     print(f"Probability = {p}")
    #     print(str(ch.bj_round))
    #     print()
    # print("-" * 32)


In [ ]:
# this_node.has_completed_tree = False
# this_node.has_built_children = False
# this_node.children_prob = []
# this_node.children = []

In [ ]:
print(this_node.children[1].children[-1].bj_round)

In [ ]:
for i in range(100):
    this_node.resample_player_cards()
    children_of_interest = this_node.children[1].children[-1].children

    # if this_node.children_prob[1] == 0:
    #    continue
    
    # print(this_node.get_value())
    # print([f"{ch.get_value():.2f}" for ch in this_node.children])
    print(this_node.children_prob)

In [ ]:
this_node.resample_player_cards()

for ch in this_node.children:
    print(ch.get_value())
    print(str(ch.bj_round))
    print()

In [ ]:

print(this_node.children[0].get_value())

In [ ]:
print(this_node.children)